# Spectral feature ablation under LOTO (Supplementary Table S6)

Compares PSR-50, Spectral-48, and Combined-98 feature sets under the strict leave-one-task-out protocol.

**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).


In [ ]:
import os, glob, warnings, time
import numpy as np
import pandas as pd
import h5py
import scipy.stats as sst
import scipy.signal as ssig
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import IsolationForest

warnings.filterwarnings("ignore")
np.random.seed(42)

# Paths
ROOT = r"D:\Research\RCM"
BASE = os.path.join(ROOT, "Lab_Data")
OUT  = os.path.join(ROOT, "Processed_Data")
os.makedirs(OUT, exist_ok=True)
OUT_CSV = os.path.join(OUT, "NB14v5_R14_spectral_comparison_nboot10k.csv")

# Constants
TASK_PAYLOAD = {"T1": 0.0, "T2": 1.0, "T3": 3.0, "T4": 2.0}
PAYLOAD_COM  = np.array([0.0, 0.0, 0.05])
GRAVITY      = np.array([0.0, 0.0, -9.81])
RATE         = 125
SUBSAMPLE    = 4
MIN_SAMP     = 200
N_BOOT       = 10000   # Match V5 NB10d (cell 43): N_BOOT = 10000 for BCa quantile precision
FIT_RATE     = RATE / SUBSAMPLE  # 31.25 Hz subsampled fitting rate
NYQUIST      = FIT_RATE / 2.0    # 15.625 Hz

UR5_DH_A     = [0, -0.42500, -0.39225, 0, 0, 0]
UR5_DH_D     = [0.089159, 0, 0, 0.10915, 0.09465, 0.0823]
UR5_DH_ALPHA = [np.pi/2, 0, 0, np.pi/2, -np.pi/2, 0]
UR5_MASS     = [3.7000, 8.3930, 2.2750, 1.2190, 1.2190, 0.1879]
UR5_COM      = [
    [0.0,     -0.02561,  0.00193],
    [0.21250,  0.0,      0.11336],
    [0.11993,  0.0,      0.02650],
    [0.0,     -0.00180,  0.01634],
    [0.0,      0.00180,  0.01634],
    [0.0,      0.0,     -0.00116],
]

def dh_transform(a, d, alpha, theta):
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [0,    sa,    ca,    d   ],
        [0,    0,     0,     1   ]
    ])

def gravity_torque(q, payload_mass=0.0):
    T = [np.eye(4)]
    for i in range(6):
        T.append(T[-1] @ dh_transform(
            UR5_DH_A[i], UR5_DH_D[i], UR5_DH_ALPHA[i], q[i]))
    com_world = [(T[i+1] @ np.array([*UR5_COM[i], 1.0]))[:3] for i in range(6)]
    masses = list(UR5_MASS)
    if payload_mass > 0:
        masses.append(payload_mass)
        com_world.append((T[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
    tau_g = np.zeros(6)
    dq = 1e-6
    for i in range(6):
        qp = q.copy(); qp[i] += dq
        Tp = [np.eye(4)]
        for jj in range(6):
            Tp.append(Tp[-1] @ dh_transform(
                UR5_DH_A[jj], UR5_DH_D[jj], UR5_DH_ALPHA[jj], qp[jj]))
        for jj in range(len(masses)):
            cp = ((Tp[jj+1] @ np.array([*UR5_COM[jj], 1.0]))[:3] if jj < 6
                  else (Tp[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
            tau_g[i] += masses[jj] * GRAVITY @ ((cp - com_world[jj]) / dq)
    return tau_g

REGISTRY = {
    "T1_healthy":   ("T1_PickPlace/Healthy",   "UR5_T1_healthy_180cyc_*.h5",                  "T1", 0),
    "T2_healthy":   ("T2_Assembly/Healthy",    "UR5_T2_healthy_180cyc_*.h5",                  "T2", 0),
    "T3_healthy":   ("T3_Palletize/Healthy",   "UR5_T3_healthy_183cyc_*.h5",                  "T3", 0),
    "T4_healthy":   ("T4_BinReorient/healthy", "UR5_T4_healthy_session2_35cyc_*.h5",          "T4", 0),
    "T1_A2_0.5kg":  ("T1_PickPlace/A2", "UR5_T1_A2_0.5kg_gripper_40cyc_*.h5",                 "T1", 1),
    "T1_A2_1kg":    ("T1_PickPlace/A2", "UR5_T1_A2_1kg_gripper_40cyc_*.h5",                   "T1", 1),
    "T1_A2_2kg":    ("T1_PickPlace/A2", "UR5_T1_A2_2kg_gripper_40cyc_*.h5",                   "T1", 1),
    "T1_A3_10wraps":("T1_PickPlace/A3", "UR5_T1_A3_1band_40cyc_*.h5",                         "T1", 1),
    "T1_A3_17wraps":("T1_PickPlace/A3", "UR5_T1_A3_3bands_40cyc_*.h5",                        "T1", 1),
    "T1_A5_20mm":   ("T1_PickPlace/A5", "UR5_T1_A5_20mm_40cyc_*.h5",                          "T1", 1),
    "T1_A5_50mm":   ("T1_PickPlace/A5", "UR5_T1_A5_50mm_40cyc_*.h5",                          "T1", 1),
    "T1_A5_100mm":  ("T1_PickPlace/A5", "UR5_T1_A5_100mm_40cyc_*.h5",                         "T1", 1),
    "T2_A2_1.5kg":  ("T2_Assembly/A2", "UR5_T2_A2_1.5kg_gripper_40cyc_*.h5",                  "T2", 1),
    "T2_A2_2kg":    ("T2_Assembly/A2", "UR5_T2_A2_2kg_gripper_40cyc_*.h5",                    "T2", 1),
    "T2_A2_3kg":    ("T2_Assembly/A2", "UR5_T2_A2_3kg_gripper_40cyc_*.h5",                    "T2", 1),
    "T2_A3_7duct":  ("T2_Assembly/A3", "UR5_T2_A3_light_duct_40cyc_*.h5",                     "T2", 1),
    "T2_A3_14duct": ("T2_Assembly/A3", "UR5_T2_A3_medium_duct_40cyc_*_225508.h5",             "T2", 1),
    "T2_A5_20mm":   ("T2_Assembly/A5", "UR5_T2_A5_20mm_40cyc_*.h5",                           "T2", 1),
    "T2_A5_50mm":   ("T2_Assembly/A5", "UR5_T2_A5_50mm_40cyc_*.h5",                           "T2", 1),
    "T2_A5_100mm":  ("T2_Assembly/A5", "UR5_T2_A5_100mm_40cyc_*.h5",                          "T2", 1),
    "T3_A2_3.5kg":  ("T3_Palletize/A2", "UR5_T3_A2_3.5kg_gripper_33cyc_*.h5",                 "T3", 1),
    "T3_A2_4kg":    ("T3_Palletize/A2", "UR5_T3_A2_4kg_gripper_33cyc_*.h5",                   "T3", 1),
    "T3_A2_5kg":    ("T3_Palletize/A2", "UR5_T3_A2_4.5kg_gripper_33cyc_*.h5",                 "T3", 1),
    "T3_A3_7duct":  ("T3_Palletize/A3", "UR5_T3_A3_light_duct_33cyc_*_222457.h5",             "T3", 1),
    "T3_A3_14duct": ("T3_Palletize/A3", "UR5_T3_A3_medium_duct_33cyc_*_205648.h5",            "T3", 1),
    "T3_A5_20mm":   ("T3_Palletize/A5", "UR5_T3_A5_20mm_33cyc_*_172334.h5",                   "T3", 1),
    "T3_A5_50mm":   ("T3_Palletize/A5", "UR5_T3_A5_50mm_33cyc_*_164447.h5",                   "T3", 1),
    "T3_A5_100mm":  ("T3_Palletize/A5", "UR5_T3_A5_100mm_33cyc_*_160716.h5",                  "T3", 1),
    # T4 anomaly cycles
    "T4_A2_0.5kg":  ("T4_BinReorient/anomaly", "UR5_T4_A2_0.5kg_35cyc_*.h5",     "T4", 1),
    "T4_A2_1kg":    ("T4_BinReorient/anomaly", "UR5_T4_A2_1kg_35cyc_*.h5",       "T4", 1),
    "T4_A2_2kg":    ("T4_BinReorient/anomaly", "UR5_T4_A2_2kg_35cyc_*.h5",       "T4", 1),
    "T4_A3_7duct":  ("T4_BinReorient/anomaly", "UR5_T4_A3_7wraps_35cyc_*.h5",    "T4", 1),
    "T4_A3_14duct": ("T4_BinReorient/anomaly", "UR5_T4_A3_14wraps_35cyc_*.h5",   "T4", 1),
    "T4_A5_20mm":   ("T4_BinReorient/anomaly", "UR5_T4_A5_20mm_35cyc_*.h5",      "T4", 1),
    "T4_A5_50mm":   ("T4_BinReorient/anomaly", "UR5_T4_A5_50mm_35cyc_*.h5",      "T4", 1),
    "T4_A5_100mm":  ("T4_BinReorient/anomaly", "UR5_T4_A5_100mm_35cyc_*.h5",     "T4", 1),
}

def load_all_cycles():
    """Load every cycle from REGISTRY. Match patterns; warn on misses."""
    all_cycles = []
    for key, (subdir, pattern, task, is_anom) in REGISTRY.items():
        matches = sorted(glob.glob(os.path.join(BASE, subdir, pattern)))
        if not matches:
            print(f"  WARNING  Not found: {key}  (pattern: {pattern})")
            continue
        for path in matches:
            with h5py.File(path, "r") as f:
                cnum   = f["cycle_number"][:].astype(int).ravel()
                cur_a  = f["actual_current"][:]
                q_a    = f["actual_q"][:]
                qd_a   = f["actual_qd"][:]
            for c in np.unique(cnum[cnum > 0]):
                mask = cnum == c
                if mask.sum() >= MIN_SAMP:
                    all_cycles.append({
                        "q":           q_a[mask],
                        "qd":          qd_a[mask],
                        "current":     cur_a[mask],
                        "task":        task,
                        "is_anomaly":  is_anom,
                        "source":      key,
                    })
    return all_cycles

def precompute_torques(cycles):
    """Precompute tau_g at subsampled timesteps and cache subsample indices."""
    for cyc in cycles:
        payload = TASK_PAYLOAD[cyc["task"]]
        q_a = cyc["q"]
        N   = len(q_a)
        idx = list(range(0, N, SUBSAMPLE))
        tau_g_arr = np.zeros((len(idx), 6))
        for ti, t in enumerate(idx):
            tau_g_arr[ti] = gravity_torque(q_a[t], payload_mass=payload)
        cyc["tau_g_cached"] = tau_g_arr
        cyc["sub_idx"]      = idx

# PSR fitting (M4, all 5 regressors)
def fit_psr_M4(train_cycles):
    """Fit per-joint M4 PSR on the union of training cycles."""
    Phi = [[] for _ in range(6)]
    I   = [[] for _ in range(6)]
    for cyc in train_cycles:
        qd_a, cur = cyc["qd"], cyc["current"]
        tau_g = cyc["tau_g_cached"]
        for ti, t in enumerate(cyc["sub_idx"]):
            for j in range(6):
                qdd_j = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                         if 0 < t < len(qd_a) - 1 else 0.0)
                Phi[j].append([tau_g[ti, j], qd_a[t, j],
                               np.sign(qd_a[t, j]), qdd_j, 1.0])
                I[j].append(cur[t, j])
    Phi = [np.array(p) for p in Phi]
    I   = [np.array(i) for i in I]
    w = [np.linalg.lstsq(Phi[j], I[j], rcond=None)[0] for j in range(6)]
    return w

def compute_residuals(cyc, psr_w):
    """Per-cycle residual time series:
       res[t, j] = i_j(t) - phi_j(t) @ w_j
       gr[t, j]  = i_j(t) - (w_j[0]*tau_g + w_j[4]*1)
                 = residual that would remain if only gravity+bias were fitted
    """
    qd_a, cur = cyc["qd"], cyc["current"]
    tau_g     = cyc["tau_g_cached"]
    sub_idx   = cyc["sub_idx"]
    n_sub     = len(sub_idx)
    res = np.zeros((n_sub, 6))
    gr  = np.zeros((n_sub, 6))
    for ti, t in enumerate(sub_idx):
        for j in range(6):
            qdd_j = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                     if 0 < t < len(qd_a) - 1 else 0.0)
            phi   = np.array([tau_g[ti, j], qd_a[t, j],
                              np.sign(qd_a[t, j]), qdd_j, 1.0])
            w     = psr_w[j]
            res[ti, j] = cur[t, j] - phi @ w
            gr[ti, j]  = cur[t, j] - (w[0] * tau_g[ti, j] + w[4] * 1.0)
    return res, gr

# Feature extraction — three variants
PSR_FEAT_COLS_50 = (
    [f"J{j}_{s}" for j in range(6)
     for s in ["resid_mean", "resid_std", "resid_rms", "resid_max",
               "resid_skew", "resid_kurtosis",
               "grav_resid_std", "grav_resid_rms"]]
    + ["total_resid_rms", "J1J2_resid_corr"]
)

SPECTRAL_FEAT_COLS_48 = (
    [f"J{j}_{ch}_{s}" for j in range(6)
     for ch in ["resid", "gres"]
     for s in ["spec_centroid", "spec_spread", "spec_entropy", "dom_freq"]]
)

def extract_50d_time_features(res, gr):
    f = {}
    for j in range(6):
        r = res[:, j]; g = gr[:, j]
        f[f"J{j}_resid_mean"]     = float(r.mean())
        f[f"J{j}_resid_std"]      = float(r.std())
        f[f"J{j}_resid_rms"]      = float(np.sqrt(np.mean(r**2)))
        f[f"J{j}_resid_max"]      = float(np.abs(r).max())
        f[f"J{j}_resid_skew"]     = float(sst.skew(r))
        f[f"J{j}_resid_kurtosis"] = float(sst.kurtosis(r))
        f[f"J{j}_grav_resid_std"] = float(g.std())
        f[f"J{j}_grav_resid_rms"] = float(np.sqrt(np.mean(g**2)))
    f["total_resid_rms"] = float(np.sqrt(np.mean(res**2)))
    f["J1J2_resid_corr"] = float(
        np.corrcoef(res[:, 1], res[:, 2])[0, 1] if len(res) > 2 else 0.0)
    return f

def extract_48d_spectral_features(res, gr):
    """Welch PSD spectral statistics, per channel per joint."""
    f = {}
    for j in range(6):
        for ch_name, sig in [("resid", res[:, j]), ("gres", gr[:, j])]:
            n = len(sig)
            if n < 8:
                f[f"J{j}_{ch_name}_spec_centroid"] = 0.0
                f[f"J{j}_{ch_name}_spec_spread"]   = 0.0
                f[f"J{j}_{ch_name}_spec_entropy"]  = 0.0
                f[f"J{j}_{ch_name}_dom_freq"]      = 0.0
                continue
            nperseg = min(256, max(8, n // 4))
            try:
                freqs, psd = ssig.welch(sig, fs=FIT_RATE, nperseg=nperseg,
                                        window="hann", noverlap=nperseg // 2,
                                        scaling="density")
            except Exception:
                freqs = np.linspace(0, NYQUIST, 8)
                psd   = np.ones(8) / 8.0
            psd_sum = psd.sum()
            if psd_sum <= 0 or not np.isfinite(psd_sum):
                f[f"J{j}_{ch_name}_spec_centroid"] = 0.0
                f[f"J{j}_{ch_name}_spec_spread"]   = 0.0
                f[f"J{j}_{ch_name}_spec_entropy"]  = 0.0
                f[f"J{j}_{ch_name}_dom_freq"]      = 0.0
                continue
            psd_n = psd / psd_sum
            centroid = float(np.sum(freqs * psd_n))
            spread   = float(np.sqrt(np.sum((freqs - centroid) ** 2 * psd_n)))
            entropy  = float(-np.sum(psd_n[psd_n > 0]
                                     * np.log(psd_n[psd_n > 0])) /
                             np.log(max(2, len(psd_n))))
            dom_freq = float(freqs[np.argmax(psd)])
            f[f"J{j}_{ch_name}_spec_centroid"] = centroid
            f[f"J{j}_{ch_name}_spec_spread"]   = spread
            f[f"J{j}_{ch_name}_spec_entropy"]  = entropy
            f[f"J{j}_{ch_name}_dom_freq"]      = dom_freq
    return f

def extract_features_all_variants(cycles, psr_w):
    """For every cycle, compute residuals once and extract all three feature sets."""
    rows_50, rows_48, rows_98 = [], [], []
    meta = []
    for cyc in cycles:
        res, gr = compute_residuals(cyc, psr_w)
        f50 = extract_50d_time_features(res, gr)
        f48 = extract_48d_spectral_features(res, gr)
        f98 = {**f50, **f48}
        rows_50.append(f50); rows_48.append(f48); rows_98.append(f98)
        meta.append({"task": cyc["task"], "is_anomaly": cyc["is_anomaly"],
                     "source": cyc["source"]})
    meta_df = pd.DataFrame(meta)
    df50 = pd.concat([pd.DataFrame(rows_50), meta_df], axis=1)
    df48 = pd.concat([pd.DataFrame(rows_48), meta_df], axis=1)
    df98 = pd.concat([pd.DataFrame(rows_98), meta_df], axis=1)
    return df50, df48, df98

# Detectors
def zscore_scores(Xtr, Xte):
    mu = Xtr.mean(0); sg = Xtr.std(0) + 1e-8
    return np.abs((Xte - mu) / sg).mean(1)

def iforest_scores(Xtr, Xte, seed=42):
    # n_estimators=200, contamination=0.05, random_state=42
    clf = IsolationForest(n_estimators=200, contamination=0.05,
                          random_state=seed, n_jobs=1)
    clf.fit(Xtr)
    return -clf.decision_function(Xte)

# BCa bootstrap AUROC
def bootstrap_auroc_bca(y_true, y_score, n_boot=N_BOOT, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    auroc_obs = roc_auc_score(y_true, y_score)
    boot = np.zeros(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]; ys = y_score[idx]
        boot[b] = roc_auc_score(yt, ys) if 0 < yt.sum() < n else auroc_obs
    prop = np.clip(np.mean(boot < auroc_obs), 1e-6, 1 - 1e-6)
    z0 = sst.norm.ppf(prop)
    jack = np.zeros(n)
    for i in range(n):
        idx_j = np.concatenate([np.arange(i), np.arange(i + 1, n)])
        yt_j = y_true[idx_j]; ys_j = y_score[idx_j]
        jack[i] = (roc_auc_score(yt_j, ys_j)
                   if 0 < yt_j.sum() < len(yt_j) else auroc_obs)
    jm = jack.mean()
    num = np.sum((jm - jack) ** 3)
    den = 6.0 * (np.sum((jm - jack) ** 2) ** 1.5)
    a = num / den if den != 0 else 0.0
    ci = {}
    for label, za in [("lo", sst.norm.ppf(0.025)), ("hi", sst.norm.ppf(0.975))]:
        p = sst.norm.cdf(z0 + (z0 + za) / (1 - a * (z0 + za)))
        ci[label] = float(np.quantile(boot, np.clip(p, 0.001, 0.999)))
    return float(auroc_obs), ci["lo"], ci["hi"]

# MAIN
print("=" * 75)
print("NB14 -- Spectral feature comparison under LOTO (R1.4)")
print("=" * 75)

print("\n[Step 1] Loading HDF5 data (healthy + anomaly cycles, all 4 tasks)...")
t0 = time.perf_counter()
all_cycles = load_all_cycles()
print(f"  Total cycles loaded: {len(all_cycles)} in {time.perf_counter()-t0:.1f}s")
for t in ["T1", "T2", "T3", "T4"]:
    nh = sum(1 for c in all_cycles if c["task"] == t and c["is_anomaly"] == 0)
    na = sum(1 for c in all_cycles if c["task"] == t and c["is_anomaly"] == 1)
    print(f"    {t}: {nh} healthy, {na} anomaly")

print("\n[Step 2] Precomputing gravity torques...")
t0 = time.perf_counter()
precompute_torques(all_cycles)
print(f"  Done in {time.perf_counter()-t0:.1f}s")

# Step 3a: 3-fold T3 ANCHOR
# (Train on T1+T2 healthy only
print("\n[Step 3a] Anchor verification: 3-fold T3 LOTO with PSR-50 + Z-Score...")
print("           Target: V5 Table 5 PSR Z-Score T3 AUROC = 0.648")

anchor_tr_tasks = ["T1", "T2"]    # 3-fold protocol training set for test=T3
anchor_te_task  = "T3"
anchor_tr_cycs  = [c for c in all_cycles if c["task"] in anchor_tr_tasks]
anchor_te_cycs  = [c for c in all_cycles if c["task"] == anchor_te_task]
anchor_tr_healthy = [c for c in anchor_tr_cycs if c["is_anomaly"] == 0]

t_anchor = time.perf_counter()
anchor_psr_w = fit_psr_M4(anchor_tr_healthy)
anchor_df_tr_50, _, _ = extract_features_all_variants(anchor_tr_healthy, anchor_psr_w)
anchor_df_te_50, _, _ = extract_features_all_variants(anchor_te_cycs, anchor_psr_w)
anchor_Xtr = anchor_df_tr_50[PSR_FEAT_COLS_50].values
anchor_Xte = anchor_df_te_50[PSR_FEAT_COLS_50].values
anchor_yte = anchor_df_te_50["is_anomaly"].values
if not np.isfinite(anchor_Xtr).all() or not np.isfinite(anchor_Xte).all():
    col_mean = np.nanmean(anchor_Xtr, axis=0)
    col_mean = np.where(np.isfinite(col_mean), col_mean, 0.0)
    anchor_Xtr = np.where(np.isfinite(anchor_Xtr), anchor_Xtr, col_mean)
    anchor_Xte = np.where(np.isfinite(anchor_Xte), anchor_Xte, col_mean)

anchor_scores = zscore_scores(anchor_Xtr, anchor_Xte)
anchor_auroc, anchor_lo, anchor_hi = bootstrap_auroc_bca(anchor_yte, anchor_scores)
anchor_time = time.perf_counter() - t_anchor

print(f"  3-fold T3 anchor: AUROC = {anchor_auroc:.4f} "
      f"[{anchor_lo:.4f}, {anchor_hi:.4f}] in {anchor_time:.1f}s")
print(f"  V5 Table 5 reports: T3 PSR Z-Score = 0.648 [0.594, 0.698]")
anchor_delta = anchor_auroc - 0.648
anchor_status = "MATCH" if abs(anchor_delta) < 0.005 else ("CLOSE" if abs(anchor_delta) < 0.02 else "MISMATCH")
print(f"  Delta: {anchor_delta:+.4f}  [{anchor_status}]")

if anchor_status == "MISMATCH":
    print()
    print("  WARNING: Anchor mismatch. Pipeline drift detected before variant")
    print("           comparison. Subsequent variant results should NOT be cited.")
    print("           Continuing with the 4-fold variant comparison for diagnostic")
    print("           purposes only.")

# For test = T1: train on T2+T3 healthy (3-fold protocol)
# For test = T2: train on T1+T3 healthy (3-fold protocol)
# For test = T3: train on T1+T2 healthy (3-fold protocol)
# For test = T4: train on T1+T2+T3 healthy (4-fold extension)
# T3 anchor in Step 3a confirms byte-consistency for this protocol.
TASKS = ["T1", "T2", "T3", "T4"]
HYBRID_TRAINING = {
    "T1": ["T2", "T3"],          # 3-fold: train on T2, T3 only
    "T2": ["T1", "T3"],          # 3-fold: train on T1, T3 only
    "T3": ["T1", "T2"],          # 3-fold: train on T1, T2 only
    "T4": ["T1", "T2", "T3"],    # 4-fold extension: train on T1, T2, T3
}
print("\n[Step 3b] Hybrid LOTO matching V5 Table 4 protocol...")
print("           (3-fold for T1/T2/T3; 4-fold extension for T4)")
results = []

for test_task in TASKS:
    tr_task_list = HYBRID_TRAINING[test_task]
    print(f"\n  Fold {test_task} (train on {'+'.join(tr_task_list)}):")
    tr_cycs = [c for c in all_cycles if c["task"] in tr_task_list]
    te_cycs = [c for c in all_cycles if c["task"] == test_task]
    tr_healthy = [c for c in tr_cycs if c["is_anomaly"] == 0]

    t_fit = time.perf_counter()
    psr_w = fit_psr_M4(tr_healthy)
    print(f"    PSR fit: {len(tr_healthy)} healthy cycles, "
          f"{time.perf_counter()-t_fit:.1f}s")

    t_feat = time.perf_counter()
    df_tr_50, df_tr_48, df_tr_98 = extract_features_all_variants(
        tr_healthy, psr_w)
    df_te_50, df_te_48, df_te_98 = extract_features_all_variants(
        te_cycs, psr_w)
    print(f"    Feature extraction: train {len(df_tr_50)} cycles, "
          f"test {len(df_te_50)} cycles, "
          f"{time.perf_counter()-t_feat:.1f}s")

    y_te = df_te_50["is_anomaly"].values

    for variant_name, df_tr, df_te, feat_cols in [
        ("PSR-50",        df_tr_50, df_te_50, PSR_FEAT_COLS_50),
        ("Spectral-48",   df_tr_48, df_te_48, SPECTRAL_FEAT_COLS_48),
        ("Combined-98",   df_tr_98, df_te_98, PSR_FEAT_COLS_50 + SPECTRAL_FEAT_COLS_48),
    ]:
        Xtr = df_tr[feat_cols].values
        Xte = df_te[feat_cols].values
        # Replace any nan/inf with column means from training to be safe
        if not np.isfinite(Xtr).all() or not np.isfinite(Xte).all():
            col_mean = np.nanmean(Xtr, axis=0)
            col_mean = np.where(np.isfinite(col_mean), col_mean, 0.0)
            Xtr = np.where(np.isfinite(Xtr), Xtr, col_mean)
            Xte = np.where(np.isfinite(Xte), Xte, col_mean)

        for det_name, det_fn in [
            ("Z-Score",  zscore_scores),
            ("IsoForest", iforest_scores),
        ]:
            scores = det_fn(Xtr, Xte)
            auroc, lo, hi = bootstrap_auroc_bca(y_te, scores)
            print(f"    {variant_name:14s} + {det_name:9s}: "
                  f"AUROC = {auroc:.4f}  [{lo:.4f}, {hi:.4f}]")
            results.append({
                "fold":      test_task,
                "variant":   variant_name,
                "detector":  det_name,
                "auroc":     auroc,
                "ci_lo":     lo,
                "ci_hi":     hi,
                "n_train":   len(df_tr),
                "n_test":    len(df_te),
                "n_features": len(feat_cols),
            })

df_results = pd.DataFrame(results)
df_results.to_csv(OUT_CSV, index=False)
print(f"\nWrote: {OUT_CSV}  ({len(df_results)} rows)")

# Mean across folds + Table 5 anchor verification
print("\n" + "=" * 75)
print("Summary: Mean AUROC across 4 folds")
print("=" * 75)

pivot = df_results.pivot_table(index="variant", columns="detector",
                               values="auroc", aggfunc="mean")
pivot = pivot.reindex(["PSR-50", "Spectral-48", "Combined-98"])
print(pivot.to_string(float_format=lambda x: f"{x:.4f}"))

print("\n" + "=" * 75)
print("Hybrid-LOTO PSR-50 reproduction (matches V5 Table 4 protocol)")
print("=" * 75)
print("\nPSR-50 + Z-Score per fold:")
psr50_z = df_results[(df_results["variant"] == "PSR-50") &
                     (df_results["detector"] == "Z-Score")]
v5_psr_z = {"T1": 0.966, "T2": 0.904, "T3": 0.648, "T4": 0.851}
for _, row in psr50_z.iterrows():
    v5 = v5_psr_z[row["fold"]]
    d  = row["auroc"] - v5
    status = "MATCH" if abs(d) < 0.005 else ("CLOSE" if abs(d) < 0.02 else "MISMATCH")
    print(f"  {row['fold']}: {row['auroc']:.4f}  [{row['ci_lo']:.4f}, {row['ci_hi']:.4f}]"
          f"  V5={v5:.3f}  diff={d:+.4f}  [{status}]")
mean_psr50_z = psr50_z["auroc"].mean()
v5_mean_z = sum(v5_psr_z.values())/4
print(f"  Mean: {mean_psr50_z:.4f}  V5 Table 4 mean = {v5_mean_z:.4f}")

print("\nPSR-50 + IsoForest per fold:")
psr50_i = df_results[(df_results["variant"] == "PSR-50") &
                     (df_results["detector"] == "IsoForest")]
v5_psr_i = {"T1": 0.897, "T2": 0.928, "T3": 0.635, "T4": 0.795}
for _, row in psr50_i.iterrows():
    v5 = v5_psr_i[row["fold"]]
    d  = row["auroc"] - v5
    status = "MATCH" if abs(d) < 0.005 else ("CLOSE" if abs(d) < 0.02 else "MISMATCH")
    print(f"  {row['fold']}: {row['auroc']:.4f}  [{row['ci_lo']:.4f}, {row['ci_hi']:.4f}]"
          f"  V5={v5:.3f}  diff={d:+.4f}  [{status}]")
mean_psr50_i = psr50_i["auroc"].mean()
v5_mean_i = sum(v5_psr_i.values())/4
print(f"  Mean: {mean_psr50_i:.4f}  V5 Table 4 mean = {v5_mean_i:.4f}")

print("\n" + "=" * 75)
print("FINAL ANCHOR STATUS (3-fold T3 reproduction)")
print("=" * 75)
print(f"  3-fold T3 anchor: reproduced {anchor_auroc:.4f}, "
      f"V5 Table 5 T3 = 0.648, diff {anchor_delta:+.4f}  [{anchor_status}]")
print()
if anchor_status == "MATCH":
    print("RESULT: Pipeline byte-consistent with V5 Table 5 (3-fold T3 within 0.005).")
    print("        Variant comparison values above are derived from the same fitting")
    print("        pipeline that produced V5 Table 5 and can be cited.")
elif anchor_status == "CLOSE":
    print("RESULT: Pipeline close to V5 Table 5 (within 0.02 but not 0.005).")
    print("        Investigate residual differences before citing variant comparison.")
else:
    print("RESULT: Pipeline MISMATCH with V5 Table 5. Variant comparison should NOT")
    print("        be cited until the drift is resolved.")
print("=" * 75)